<a href="https://colab.research.google.com/github/JoshuneArriaga/Datos-Masivos/blob/main/Tarea_1_Datos_Masivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Descripción del conjunto de datos

Se utiliza el dump oficial de Wikipedia en ingles en formato XML, descargado directamente de [dumps.wikimedia.org](https://dumps.wikimedia.org/enwiki/latest/).

El dump completo contiene más de **6.8** millones de artículos y pesa 24.2 GB comprimido. Wikimedia divide este archivo en múltiples partes (multistream) para facilitar su descarga y procesamiento. Se utilizara para este proyecto solo la **parte 1** (~280 MB comprimido, ~41,000 artículos), que ya representa un volumen considerable de datos semi-estructurados.

###¿Por qué este conjunto de datos?

1. Contiene un numero masivo de datos, aun incluso una partición contiene decenas de artículos con texto completo.
2. Contiene datos semi estructurados
3. Los dumps de Wikipedia son uno de los datasets abiertos mas utilizados en la industria.
4. El mismo código puede escalar al dump completo de 24 GB en un clúster real de Spark.

### Estructura del XML de Wikipedia

```xml
<page>
  <title>Nombre del artículo</title>
  <ns>0</ns>                          <!-- 0 = artículo principal -->
  <id>12345</id>
  <revision>
    <id>67890</id>
    <timestamp>2025-12-15T10:30:00Z</timestamp>
    <contributor><username>Editor</username></contributor>
    <text bytes="5432">...contenido en wikitext...</text>
  </revision>
</page>
```

## 2. Instalación del entorno Spark en Google Colab

In [4]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession.builder.appName("Mi primera sesion").getOrCreate()

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
82 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
os.environ['SPARK_HOME'] = '/content/spark-3.2.1-bin-hadoop3.2'
findspark.init()

## 3. Descarga del dump de Wikipedia

Descargamos la **parte 1** del dump multistream directamente de `dumps.wikimedia.org`.  

In [6]:
import os

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

#Parte 1 del dump multistream (Feb 2026)
DUMP_URL = 'https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2'
DUMP_FILE = os.path.join(DATA_DIR, 'enwiki-part1.xml.bz2')

if not os.path.exists(DUMP_FILE):
    print('Descargando dump de Wikipedia (parte 1, ~280 MB)...')
    !wget -q --show-progress -O {DUMP_FILE} {DUMP_URL}
    print('Descarga completada.')
else:
    print('Archivo ya descargado.')

size_mb = os.path.getsize(DUMP_FILE) / (1024**2)
print(f'Tamaño del archivo: {size_mb:.1f} MB')

Descargando dump de Wikipedia (parte 1, ~280 MB)...
/content/data/enwik 100%[===================>] 281.49M  4.74MB/s    in 60s     
Descarga completada.
Tamaño del archivo: 281.5 MB


In [12]:
# Función para parsear el dump XML de Wikipedia
import bz2
import xml.etree.ElementTree as ET
import mwparserfromhell
import re
import time

def detect_namespace(filepath):
    """Detecta el namespace XML del dump de Wikipedia."""
    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            match = re.search(r'xmlns="([^"]+)"', line)
            if match:
                ns = match.group(1)
                print(f'Namespace detectado: {ns}')
                return '{' + ns + '}'
    return ''

def wikitext_to_plain(wikicode):
    try:
        return mwparserfromhell.parse(wikicode).strip_code().strip()
    except:
        return wikicode

def extract_categories(wikitext):
    return re.findall(r'\[\[Category:([^\]|]+)', wikitext)

def count_wikilinks(wikitext):
    return len(re.findall(r'\[\[[^:][^\]]*\]\]', wikitext))

def count_sections(wikitext):
    return len(re.findall(r'^==[^=]', wikitext, re.MULTILINE))

def count_references(wikitext):
    return len(re.findall(r'<ref[\s>]', wikitext))

def count_external_links(wikitext):
    return len(re.findall(r'\[https?://', wikitext))


def parse_dump(filepath, max_articles=None):
    # Detectar namespace automáticamente
    NS = detect_namespace(filepath)

    articles = []
    count = 0
    skipped = 0
    t0 = time.time()

    print('Parseando dump XML...')

    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for event, elem in ET.iterparse(f, events=('end',)):
            if elem.tag != f'{NS}page':
                continue

            ns_tag = elem.find(f'{NS}ns')
            if ns_tag is None or ns_tag.text != '0':
                elem.clear()
                skipped += 1
                continue

            revision = elem.find(f'{NS}revision')
            if revision is None:
                elem.clear()
                continue

            text_elem = revision.find(f'{NS}text')
            wikitext = text_elem.text if (text_elem is not None and text_elem.text) else ''

            if wikitext.strip().upper().startswith('#REDIRECT'):
                elem.clear()
                skipped += 1
                continue

            title_elem = elem.find(f'{NS}title')
            id_elem    = elem.find(f'{NS}id')
            ts_elem    = revision.find(f'{NS}timestamp')

            title     = title_elem.text if title_elem is not None else ''
            page_id   = int(id_elem.text) if id_elem is not None else 0
            timestamp = ts_elem.text if ts_elem is not None else ''

            plain_text = wikitext_to_plain(wikitext)
            categories = extract_categories(wikitext)

            articles.append({
                'id':              page_id,
                'title':           title,
                'text':            plain_text[:5000],
                'text_length':     len(plain_text),
                'word_count':      len(plain_text.split()),
                'timestamp':       timestamp,
                'num_categories':  len(categories),
                'categories':      '|'.join(categories[:20]),
                'num_links':       count_wikilinks(wikitext),
                'num_sections':    count_sections(wikitext),
                'num_references':  count_references(wikitext),
                'num_ext_links':   count_external_links(wikitext),
                'wikitext_bytes':  len(wikitext.encode('utf-8')),
            })

            count += 1
            if count % 5000 == 0:
                elapsed = time.time() - t0
                print(f'  {count:>6,} artículos  ({elapsed:.0f}s)')

            elem.clear()

            if max_articles and count >= max_articles:
                break

    elapsed = time.time() - t0
    print(f'\nTerminado: {count:,} artículos, {skipped:,} omitidos  ({elapsed:.0f}s)')
    return articles

In [13]:
articles = parse_dump(DUMP_FILE, max_articles=50_000)

Namespace detectado: http://www.mediawiki.org/xml/export-0.11/
Parseando dump XML...
   5,000 artículos  (338s)
  10,000 artículos  (693s)
  15,000 artículos  (1040s)
  20,000 artículos  (1295s)

Terminado: 21,009 artículos, 6,351 omitidos  (1328s)


In [14]:
# Crear DataFrame de Spark y guardar como Parquet
schema = T.StructType([
    T.StructField('id',              T.IntegerType(),  False),
    T.StructField('title',           T.StringType(),   True),
    T.StructField('text',            T.StringType(),   True),
    T.StructField('text_length',     T.IntegerType(),  True),
    T.StructField('word_count',      T.IntegerType(),  True),
    T.StructField('timestamp',       T.StringType(),   True),
    T.StructField('num_categories',  T.IntegerType(),  True),
    T.StructField('categories',      T.StringType(),   True),
    T.StructField('num_links',       T.IntegerType(),  True),
    T.StructField('num_sections',    T.IntegerType(),  True),
    T.StructField('num_references',  T.IntegerType(),  True),
    T.StructField('num_ext_links',   T.IntegerType(),  True),
    T.StructField('wikitext_bytes',  T.IntegerType(),  True),
])

df = spark.createDataFrame(articles, schema=schema)

# Agregar columnas derivadas del timestamp
df = (df
    .withColumn('last_edit_date', F.to_date('timestamp'))
    .withColumn('last_edit_year', F.year('last_edit_date'))
    .withColumn('last_edit_month', F.month('last_edit_date'))
)

# Guardar como Parquet
PARQUET_PATH = os.path.join(DATA_DIR, 'wikipedia_en.parquet')
df.write.mode('overwrite').parquet(PARQUET_PATH)

print(f'DataFrame guardado en: {PARQUET_PATH}')
print(f'Artículos: {df.count():,}')

DataFrame guardado en: /content/data/wikipedia_en.parquet
Artículos: 21,009


In [15]:
# Cargar desde Parquet
wiki = spark.read.parquet(PARQUET_PATH)

print(f'Total de artículos: {wiki.count():,}')
print(f'Columnas: {len(wiki.columns)}')
print()
wiki.printSchema()

Total de artículos: 21,009
Columnas: 16

root
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- text_length: integer (nullable = true)
 |-- word_count: integer (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- num_categories: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- num_links: integer (nullable = true)
 |-- num_sections: integer (nullable = true)
 |-- num_references: integer (nullable = true)
 |-- num_ext_links: integer (nullable = true)
 |-- wikitext_bytes: integer (nullable = true)
 |-- last_edit_date: date (nullable = true)
 |-- last_edit_year: integer (nullable = true)
 |-- last_edit_month: integer (nullable = true)



## 4. Análisis de datos


In [16]:
wiki.select('id','title','word_count','num_categories','num_links','num_references','last_edit_year') \
    .show(15, truncate=35)

+---+-----------------------------------+----------+--------------+---------+--------------+--------------+
| id|                              title|word_count|num_categories|num_links|num_references|last_edit_year|
+---+-----------------------------------+----------+--------------+---------+--------------+--------------+
| 12|                          Anarchism|      6818|            13|      648|            14|          2026|
| 39|                             Albedo|      4487|            10|      181|           138|          2026|
|290|                                  A|      2065|             2|      322|            24|          2026|
|303|                            Alabama|     13600|             8|      955|           331|          2026|
|305|                           Achilles|      8867|            11|      603|           123|          2026|
|307|                    Abraham Lincoln|     11906|            43|      624|            92|          2026|
|308|                       

In [17]:
#Artículos sustanciales (más de 500 palabras)
sustanciales = wiki.filter(F.col('word_count') > 500)
total = wiki.count()

print(f'Articulos con mas de 500 palabras: {sustanciales.count():,} de {total:,} '
      f'({sustanciales.count()/total*100:.1f}%)')

sustanciales.select('title','word_count','num_references') \
    .orderBy(F.desc('word_count')).show(10, truncate=45)

Articulos con mas de 500 palabras: 17,606 de 21,009 (83.8%)
+---------------------------------------------+----------+--------------+
|                                        title|word_count|num_references|
+---------------------------------------------+----------+--------------+
|                         History of Australia|     36136|           622|
|                           History of Austria|     33182|            77|
|                   Foreign relations of India|     32778|           799|
|                                        2000s|     32584|           500|
|                           History of Germany|     31547|           423|
|                    Foreign relations of Iraq|     30893|          1140|
|                           Sardinian language|     30737|           670|
|List of historical films set in Near Easte...|     29830|            12|
|                                Ion Antonescu|     29737|           860|
|                      List of Dune characters|     

In [18]:
#Summary stats
cols_num = ['word_count','text_length','num_categories','num_links',
            'num_sections','num_references','num_ext_links','wikitext_bytes']

wiki.select(cols_num).describe().show()

+-------+------------------+------------------+------------------+-----------------+-----------------+-----------------+------------------+------------------+
|summary|        word_count|       text_length|    num_categories|        num_links|     num_sections|   num_references|     num_ext_links|    wikitext_bytes|
+-------+------------------+------------------+------------------+-----------------+-----------------+-----------------+------------------+------------------+
|  count|             21009|             21009|             21009|            21009|            21009|            21009|             21009|             21009|
|   mean| 3722.642772145271| 24032.15245847018|  8.31624541863011|252.3518968061307|8.436860393164833|71.72278547289257| 7.953448522061973|50137.228759103244|
| stddev|3808.0109357817946|24534.660006139195|12.271795168452192|291.6503797255298|4.248697749232599|99.34345870101855|15.913029517000203|  55872.4308773424|
|    min|                 0|                 0

In [19]:
print('Articulos por año de ultima edicion:')
wiki.groupBy('last_edit_year') \
    .agg(
        F.count('*').alias('articulos'),
        F.round(F.avg('word_count'), 0).alias('avg_palabras'),
        F.round(F.avg('num_references'), 1).alias('avg_refs'),
        F.round(F.avg('num_links'), 0).alias('avg_enlaces'),
    ) \
    .orderBy('last_edit_year') \
    .show(30)

Articulos por año de ultima edicion:
+--------------+---------+------------+--------+-----------+
|last_edit_year|articulos|avg_palabras|avg_refs|avg_enlaces|
+--------------+---------+------------+--------+-----------+
|          2012|        1|        26.0|     0.0|        3.0|
|          2013|        3|        64.0|     0.0|        6.0|
|          2014|        1|        48.0|     0.0|        6.0|
|          2015|        6|        48.0|     0.0|        5.0|
|          2016|        5|        66.0|     0.0|        7.0|
|          2017|        6|        45.0|     0.0|        5.0|
|          2018|       11|       102.0|     0.1|       12.0|
|          2019|       28|        68.0|     0.1|        7.0|
|          2020|       34|        89.0|     0.3|        8.0|
|          2021|       57|        90.0|     0.2|       10.0|
|          2022|       78|       114.0|     0.6|       13.0|
|          2023|      152|       184.0|     1.3|       17.0|
|          2024|      545|       357.0|     3.3|